In [0]:
#Read data from source
from pyspark.sql.functions import col,when,current_timestamp,current_date,to_date
stocks_df=spark.read.json('/Volumes/stocks/landing/stocks_v/stock_prices_day*.json')
stocks_df.writeTo('stocks.bronze.t_stocks').createOrReplace()


In [0]:
#Read bronze table and do transformations
bronze_df=spark.read.table('stocks.bronze.t_stocks')
bronze_df=bronze_df.withColumn("trading_date",col('trading_date').cast('date'))
bronze_df=bronze_df.withColumn('ISRT_TMS', current_timestamp())
bronze_df=bronze_df.filter(bronze_df['status'] == 'ACTIVE')


In [0]:
#create country data
cty_df=spark.createDataFrame(data=(['AAPL','IND'],['AMZN','UK'],['GOOGL','USA'],['META','CAN'],['MSFT','JPN'],['TSLA','IND']),schema=['stock_id','country'])

In [0]:
#prepare silver table data
bronze_df=bronze_df.join(cty_df, on='stock_id',how='left')

#check to not load duplicate data incase of rerun
silver_s=spark.table('stocks.silver.s_stocks')
silver_s=silver_s.filter(to_date(col('ISRT_TMS'))== current_date())
bronze_final=bronze_df.join(silver_s,on=['stock_id','trading_date'],how='leftanti')

In [0]:
#write data to silver table
bronze_final.write.mode('append').saveAsTable('stocks.silver.s_stocks')

In [0]:
src = "/Volumes/stocks/landing/stocks_v/"
dst = "/Volumes/stocks/landing/stocks_processed/"

for f in dbutils.fs.ls(src):
    dbutils.fs.mv(f.path, dst, True)